In [ ]:
import os
import json
import httpx
import asyncpg
import asyncio
from typing import Annotated, List
from langgraph.graph.message import add_messages          # ✅ BUG FIX IMPORT
from langchain_core.messages import (
    BaseMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage
)
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing_extensions import TypedDict

load_dotenv()

# ─────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────
SESSION_FARMER_ID = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb"
MANDI_JSON_PATH   = r"D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"
AGMARKNET_URL     = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"
OPEN_METEO_URL    = "https://api.open-meteo.com/v1/forecast"   # ← NEW

# ─────────────────────────────────────────────
#  SYSTEM PROMPT  (updated with 3rd tool)
# ─────────────────────────────────────────────
SYSTEM_INSTRUCTIONS = f"""You are KisanSaathi, a friendly agricultural assistant for Indian farmers.
You are helping Farmer ID: {SESSION_FARMER_ID}.

You have EXACTLY THREE tools:
  • list_mandis(crop)                    — Lists APMCs in the farmer's district.
  • fetch_crop_price(mandi_name, crop)   — Fetches today's live mandi price.
  • get_weather(days)                    — Fetches weather forecast for the farmer's field.

══════════════════════════════════════════════
PRICE QUERY FLOW (3 strict steps)
══════════════════════════════════════════════
STEP 1 — If no crop mentioned → Ask for crop. Do NOT call any tool.
STEP 2 — Crop known, no mandi → Call list_mandis(crop) ONCE. Show list. Wait for selection.
STEP 3 — Mandi selected → Call fetch_crop_price(mandi_name, crop) ONCE. Show result. STOP.

══════════════════════════════════════════════
WEATHER QUERY FLOW (1 step)
══════════════════════════════════════════════
If farmer asks about mausam/barish/garmi/temperature/forecast:
  → Extract how many days (aaj=1, kal=2, teen din=3, hafte=7). Default=3.
  → Call get_weather(days=<number>) EXACTLY ONCE.
  → Show the result. STOP.

══════════════════════════════════════════════
HARD RULES
══════════════════════════════════════════════
✗ NEVER call list_mandis more than once per user turn.
✗ NEVER call fetch_crop_price more than once per user turn.
✗ NEVER call get_weather more than once per user turn.
✗ If a tool already returned data this turn, DO NOT call it again.
✓ Respond in Hinglish (Hindi + English mix)."""


# ─────────────────────────────────────────────
#  TOOL 1 — list_mandis
# ─────────────────────────────────────────────
@tool
async def list_mandis(crop: str) -> str:
    """
    Returns a list of APMC mandis in the farmer's district for a given crop.
    Call ONLY when you have the crop name but no mandi selected yet.
    """
    try:
        conn = await asyncpg.connect(os.getenv("DATABASE_URL"))
        row  = await conn.fetchrow(
            "SELECT state_name, dist_name FROM farmers WHERE id = $1",
            SESSION_FARMER_ID
        )
        await conn.close()
    except Exception as e:
        return f"Database Error: {e}"

    if not row:
        return "Error: Farmer not found in database."

    state    = row["state_name"]
    district = row["dist_name"]

    try:
        with open(MANDI_JSON_PATH, "r", encoding="utf-8") as f:
            mandi_data = json.load(f)
    except FileNotFoundError:
        return "Error: Mandi master data file not found."

    available_mandis: list[str] = []
    for s_key, districts in mandi_data.items():
        if s_key.lower() == state.lower():
            for d_key, mandis in districts.items():
                if d_key.lower() == district.lower():
                    available_mandis = mandis
                    break

    if not available_mandis:
        return (
            f"Aapke district {district} ({state}) ke liye mandi data nahi hai. "
            "Kripya baad mein try karein."
        )

    mandi_list = "\n".join(f"  {i+1}. {m}" for i, m in enumerate(available_mandis))
    return (
        f"📍 {district}, {state} ke available mandis ({crop} ke liye):\n"
        f"{mandi_list}\n\n"
        f"Kaunsi mandi ka bhav dekhna hai? (Number ya naam batayein)"
    )


# ─────────────────────────────────────────────
#  TOOL 2 — fetch_crop_price
# ─────────────────────────────────────────────
@tool
async def fetch_crop_price(mandi_name: str, crop: str) -> str:
    """
    Fetches today's live price for a crop at a specific APMC mandi.
    Call ONLY after the farmer has explicitly selected a mandi.
    """
    if not mandi_name or not crop:
        return "Error: Both mandi_name and crop are required."

    try:
        conn = await asyncpg.connect(os.getenv("DATABASE_URL"))
        row  = await conn.fetchrow(
            "SELECT state_name FROM farmers WHERE id = $1",
            SESSION_FARMER_ID
        )
        await conn.close()
    except Exception as e:
        return f"Database Error: {e}"

    if not row:
        return "Error: Farmer not found."

    state      = row["state_name"]
    clean_name = mandi_name.replace("APMC", "").strip()

    params = {
        "api-key":            os.getenv("AGMARKNET_API_KEY"),
        "format":             "json",
        "filters[state]":     state,
        "filters[market]":    clean_name,
        "filters[commodity]": crop,
    }

    try:
        async with httpx.AsyncClient() as client:
            resp = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
    except httpx.TimeoutException:
        return "API Timeout: Government server ne respond nahi kiya. Thodi der baad try karein."

    if resp.status_code != 200:
        return f"API Error {resp.status_code}: Government server busy hai."

    records = resp.json().get("records", [])
    if not records:
        return (
            f"⚠️  {crop} ka aaj ka data {clean_name} mandi mein available nahi hai. "
            "Kal try karein ya doosri mandi chunein."
        )

    rec = records[0]
    return (
        f"✅ Live Price\n"
        f"━━━━━━━━━━━━━━━━━━━━━━━\n"
        f"🌾 Crop    : {rec['commodity']}\n"
        f"🏪 Mandi   : {rec['market']}\n"
        f"💰 Price   : ₹{rec['modal_price']} / Quintal\n"
        f"📅 Date    : {rec['arrival_date']}\n"
        f"━━━━━━━━━━━━━━━━━━━━━━━"
    )


# ─────────────────────────────────────────────
#  TOOL 3 — get_weather  ← NEW
#  Step 1: DB → fetch center_lat, center_lon from farm_fields
#  Step 2: Open-Meteo API → fetch forecast
#  Step 3: Return formatted result
# ─────────────────────────────────────────────
@tool
async def get_weather(days: int = 3) -> str:
    """
    Fetches weather forecast for the farmer's registered field location.
    Uses farmer's GPS coordinates from the database + Open-Meteo free API.
    Call this for any mausam/barish/garmi/temperature/forecast questions.
    days: number of forecast days (1=aaj, 2=kal tak, 3=teen din, 7=hafte bhar). Default=3.
    """
    # ── Step 1: Fetch lat/lon from farm_fields table ──────
    try:
        conn = await asyncpg.connect(os.getenv("DATABASE_URL"))
        row  = await conn.fetchrow(
            """
            SELECT field_name, city_name, center_lat, center_lon
            FROM   farm_fields
            WHERE  farmer_id = $1
            LIMIT  1
            """,
            SESSION_FARMER_ID
        )
        await conn.close()
    except Exception as e:
        return f"Database Error: Could not fetch field location. {e}"

    if not row:
        return (
            "⚠️  Aapka khet registered nahi hai. "
            "Pehle apna khet register karein, phir weather dekh sakte hain."
        )

    lat        = float(row["center_lat"])
    lon        = float(row["center_lon"])
    field_name = row["field_name"] or "Aapka Khet"
    city       = row["city_name"]  or ""

    # Clamp days to Open-Meteo allowed range
    days = max(1, min(days, 16))

    # ── Step 2: Call Open-Meteo API (no key needed) ───────
    params = {
        "latitude":      lat,
        "longitude":     lon,
        "daily":         "temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max",
        "forecast_days": days,
        "timezone":      "Asia/Kolkata",
    }

    try:
        async with httpx.AsyncClient() as client:
            resp = await client.get(OPEN_METEO_URL, params=params, timeout=15.0)
        resp.raise_for_status()
    except httpx.TimeoutException:
        return "⚠️  Mausam server ne respond nahi kiya. Thodi der baad try karein."
    except Exception as e:
        return f"⚠️  Weather API error: {e}"

    # ── Step 3: Format result ─────────────────────────────
    daily = resp.json().get("daily", {})
    dates = daily.get("time", [])
    t_max = daily.get("temperature_2m_max", [])
    t_min = daily.get("temperature_2m_min", [])
    rain  = daily.get("precipitation_sum", [])
    wind  = daily.get("windspeed_10m_max", [])

    if not dates:
        return "⚠️  Mausam data abhi available nahi hai. Thodi der baad try karein."

    location_line = f"{field_name}" + (f", {city}" if city else "")
    lines = [
        f"🌤️  Mausam Forecast — {location_line}",
        f"📍 GPS: ({lat:.4f}, {lon:.4f})",
        f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━",
    ]

    for i in range(min(days, len(dates))):
        rain_mm    = rain[i] if rain[i] is not None else 0.0
        wind_kmh   = wind[i] if wind[i] is not None else 0.0

        if rain_mm >= 10:
            rain_icon = f"🌧️  {rain_mm}mm (Khet mein kaam band rakhein!)"
        elif rain_mm > 0:
            rain_icon = f"🌦️  {rain_mm}mm (Halki baarish)"
        else:
            rain_icon = "☀️  Baarish nahi"

        wind_note = f" ⚠️  (Tej aandhi!)" if wind_kmh > 40 else ""

        lines.append(
            f"\n📅 {dates[i]}\n"
            f"   🌡️  Temp : {t_min[i]}°C – {t_max[i]}°C\n"
            f"   {rain_icon}\n"
            f"   💨 Hawa : {wind_kmh} km/h{wind_note}"
        )

    lines.append("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    return "\n".join(lines)


# ─────────────────────────────────────────────
#  STATE  ← BUG FIX: add_messages reducer
# ─────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]   # ✅ APPENDS, never replaces


# ─────────────────────────────────────────────
#  MODEL — bind ALL 3 tools
# ─────────────────────────────────────────────
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROK_API_KEY"),
).bind_tools([list_mandis, fetch_crop_price, get_weather])   # ← get_weather added
# ─────────────────────────────────────────────
#  GRAPH NODES
# ─────────────────────────────────────────────
async def call_model(state: AgentState) -> dict:
    response = await model.ainvoke(
        [SystemMessage(content=SYSTEM_INSTRUCTIONS)] + state["messages"]
    )
    return {"messages": [response]}


async def call_tool(state: AgentState) -> dict:
    last_msg = state["messages"][-1]
    results  = []

    for tool_call in last_msg.tool_calls:
        print(f"🛠️  Tool called: {tool_call['name']}({tool_call['args']})")

        if tool_call["name"] == "list_mandis":
            content = await list_mandis.ainvoke(tool_call["args"])
        elif tool_call["name"] == "fetch_crop_price":
            content = await fetch_crop_price.ainvoke(tool_call["args"])
        elif tool_call["name"] == "get_weather":          # ← NEW
            content = await get_weather.ainvoke(tool_call["args"])
        else:
            content = f"Unknown tool: {tool_call['name']}"

        results.append(
            ToolMessage(tool_call_id=tool_call["id"], content=str(content))
        )

    return {"messages": results}


def router(state: AgentState) -> str:
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


# ─────────────────────────────────────────────
#  BUILD GRAPH
# ─────────────────────────────────────────────
graph = StateGraph(AgentState)
graph.add_node("agent", call_model)
graph.add_node("tools", call_tool)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", router, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")
app = graph.compile()


# ─────────────────────────────────────────────
#  CHAT INTERFACE
# ─────────────────────────────────────────────
async def chat_with_farmer():
    print(f"\n🌾 KisanSaathi Active | Farmer: {SESSION_FARMER_ID}")
    print("   Tools: list_mandis | fetch_crop_price | get_weather\n")
    state = {"messages": []}

    while True:
        user_input = input("👨‍🌾 Farmer: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("quit", "exit", "band karo", "bye"):
            print("🤖 KisanSaathi: Dhanyavad! Jai Jawan Jai Kisan 🌾")
            break

        state["messages"].append(HumanMessage(content=user_input))

        result = await app.ainvoke(
            state,
            config={"recursion_limit": 8}   # 3 tools × 2 hops each = 6 needed; 8 is safe
        )
        state["messages"] = result["messages"]

        for msg in reversed(state["messages"]):
            if isinstance(msg, AIMessage) and not msg.tool_calls:
                print(f"\n🤖 KisanSaathi: {msg.content}\n")
                break


# In Jupyter:
await chat_with_farmer()

# In .py file:
# if __name__ == "__main__":
#     asyncio.run(chat_with_farmer())


🌾 KisanSaathi Active | Farmer: c59f6f44-1a98-4eaa-8cf0-3581316a32bb
   Tools: list_mandis | fetch_crop_price | get_weather

🛠️  Tool called: get_weather({'days': 3})

🤖 KisanSaathi: Aane waale teen din mausam garm rahega, taapmaan 28 se 43 degree ke beech rahega. Baarish ki sambhavana nahin hai. Hawa ki gati 19 se 20 km/h rahegi.

🛠️  Tool called: list_mandis({'crop': 'wheat'})

🤖 KisanSaathi: Ab aapko yeh mandis di gayi hain. Ab aapko apni pasand ki mandi chunni hogi. Phir main aapko us mandi mein wheat ka price bataunga.

🛠️  Tool called: list_mandis({'crop': 'wheat'})

🤖 KisanSaathi: Agra, Uttar Pradesh mein wheat ke liye yeh 6 mandis hain: 
1. Achnera APMC
2. Agra APMC
3. Fatehabad APMC
4. Fatehpur Sikri APMC
5. Jagnair APMC
6. Jarar APMC

Ab aapko inmein se kisi ek mandi ka naam ya number batana hoga, taaki main aapko us mandi mein wheat ka price bata sakun.

🛠️  Tool called: fetch_crop_price({'crop': 'wheat', 'mandi_name': 'Achnera APMC'})

🤖 KisanSaathi: Mafi chahunga, lekin ab